<!--- sf-header --->
<table align="left">
<tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fstatmike%2Fscale-forecasting%2Fmain%2Fnotebooks%2F08_job_review.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Colab Enterprise logo">
      <br>Run in<br>Colab Enterprise
    </a>
  </td>
</tr>
</table>
<br clear="left"/>

> **Run in Colab Enterprise:** click the badge to import this notebook, pick a runtime, and
> **Run all**. The Terraform-deployed templates already carry the `SF_*` run identity in their env,
> so there's no environment cell to fill in. Runs on the **`sf-main`** runtime template (Python 3.11). See
> [`docs/notebook_runtimes.md`](https://github.com/statmike/scale-forecasting/blob/main/docs/notebook_runtimes.md)
> for the per-notebook template mapping and the headless acceptance harness.


# 08 · Job review — monitor a running run, review a finished one

Every other notebook *launches* work. This one **inspects** it. Point it at a `run_id` and it answers the two questions you actually have about a run:

- **Monitor (in flight):** how far along is it? Per-family job state on its chosen runner, series done vs. the expected total, and mean fit time per family — plus a progress-bar readout. Poll it while a run is running.
- **Review (finished):** how did it *do*, in data-science detail? The best model per family and overall, the full metric panel aggregated across every series (mean + p10/p50/p90), and each ensemble's lift over the best base model — the leaderboard, the distribution, and the execution timeline of the family DAG.

The whole layer is keyed on a bare `run_id` — the run *is* its config, so this reads the run's own `raw_config` back to recover what it *planned* to do (models per family, decision metric, ensemble strategies) and measures progress against that. It uses the SDK's run-inspection API (`scale_forecasting.review`): `monitor_run` / `review_run` and their plots.

It **runs nothing** — no config, no `main.run`, no compute. It only reads the registry, so it's cheap, re-runnable, and safe to keep open beside a run that's still going.

## Get the code (cloud runtimes only)

On a cloud notebook (Colab Enterprise, Vertex Workbench) this clones or updates the repo so you're on the latest `src/`. **Skip it in a local clone** — it's a no-op guarded on the package already being importable.

In [ ]:
# Cloud bootstrap: clone + install the LOCKED dependency set (uv.lock) so
# `import scale_forecasting` resolves against the exact versions every other surface runs.
# Uses uv (Colab ships it; we install it if missing) to install the frozen lock, then
# editable-installs the package with --no-deps. Harmless locally — if it already imports, no-op.
import importlib.util
import os
import shutil
import subprocess
import sys

REPO_URL = os.environ.get("SF_REPO_URL", "https://github.com/statmike/scale-forecasting.git")
REPO_DIR = os.environ.get("SF_REPO_DIR", "scale-forecasting")

EXTRAS = []  # this notebook needs only the core (it reads the registry — no model stack)

if importlib.util.find_spec("scale_forecasting") is None:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    uv = shutil.which("uv")
    if uv is None:  # Colab Enterprise ships uv; install it if this runtime doesn't
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        uv = shutil.which("uv") or "uv"
    # Resolve the checked-in lock to a requirements file (no re-resolve), then install EXACTLY
    # that set into this kernel — the same versions the container + packed-venv are built from.
    reqs = os.path.abspath(os.path.join(REPO_DIR, "colab-requirements.txt"))
    extra_flags = [f for e in EXTRAS for f in ("--extra", e)]
    subprocess.run(
        [uv, "export", "--frozen", "--no-emit-project", "--no-hashes", "--no-dev", *extra_flags,
         "-o", reqs],
        cwd=REPO_DIR, check=True,
    )
    subprocess.run([uv, "pip", "install", "--python", sys.executable, "-r", reqs], check=True)
    subprocess.run([uv, "pip", "install", "--python", sys.executable, "-e", REPO_DIR, "--no-deps"],
                   check=True)
    # `-e` drops a .pth that only `site` reads at interpreter startup — a fresh kernel that
    # installs mid-session can't import until restart. Put src/ on the path directly so THIS
    # kernel resolves it (matches the other notebooks).
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

## Resolve the deployment (live GCP)

`Settings.resolve()` reads the `SF_*` environment — the *same* identity every writer uses. Required: `SF_PROJECT_ID`, `SF_CONNECTION`, `SF_WAREHOUSE_URI`; `SF_DATASET_ID`/`SF_REGION` default. This notebook only *reads* the run registry (via the `scale_forecasting.review` layer), so it never writes.

In [ ]:
from google.cloud import bigquery

from scale_forecasting.settings import Settings

settings = Settings.resolve()
client = bigquery.Client(project=settings.project_id)
DATASET = settings.dataset_ref
print("deployment:", DATASET, "region:", settings.region)

## Parameters — the run_id to inspect

Set `RUN_ID` to the run you want to monitor or review. Leave it `None` and the discovery cell below picks the most recent run for you — handy for a quick look. The same `run_id` drives both the Monitor and the Review sections: monitor a run while it's `RUNNING`, review it once it's `COMPLETED`.

In [ ]:
# === Parameters — edit me ===============================================
RUN_ID = None  # e.g. "all-families-100k-050eee3e0a6b"; None → auto-pick the most recent run below
# ========================================================================

## Discover recent run_ids

The most recent runs with their `status` and scale — so you can copy a `run_id` into the cell above (pick a `RUNNING` one to watch the Monitor section tick, or a `COMPLETED` one for the Review). If you left `RUN_ID = None`, the next cell auto-selects the top row.

In [ ]:
recent = client.query(
    f"SELECT run_id, created_at, status, python_runtime, n_series, n_models "
    f"FROM `{DATASET}.v_run_summary` "
    f"ORDER BY created_at DESC LIMIT 25"
).result().to_dataframe()

if RUN_ID is None and not recent.empty:
    RUN_ID = recent.iloc[0]["run_id"]
    print(f"RUN_ID not set — auto-selected most recent: {RUN_ID}")
elif RUN_ID is None:
    print("No runs found in the registry yet — run one first (see notebooks 01-04), then re-run.")
else:
    print(f"inspecting: {RUN_ID}")
recent

## Monitor — how far along is the run?

`monitor_run(run_id)` reads the run's header, its config (the *expected-work* denominator), its per-family jobs, and the landed-cell counts, and returns a `RunProgress`: the run's status, and one `FamilyProgress` per family with its runner, job status, `n_done / n_expected` cells, and mean fit time on that runner.

> Progress is **coarse by design**: the registry has no live per-series counter — cells land when a family's writer runs (often at job end), so counts step up per *job*, not per series. The per-job `status` is the primary live signal; the landed-cell counts refine it. Re-run this cell to poll a run that's still going.

In [ ]:
import pandas as pd

from scale_forecasting.review import monitor_run

progress = monitor_run(RUN_ID, settings=settings)
pct = f"{progress.fraction:.0%}" if progress.fraction is not None else "n/a"
print(f"{progress.run_id} — status={progress.status} — "
      f"{progress.n_done}/{progress.n_expected} cells ({pct}) across {progress.n_series} series")

families = pd.DataFrame([
    {
        "family": f.family,
        "runtime": f.runtime,
        "hardware": f.hardware,
        "status": f.status,
        "models": ", ".join(f.models),
        "n_done": f.n_done,
        "n_expected": f.n_expected,
        "fraction": f.fraction,
        "avg_fit_seconds": f.avg_fit_seconds,
        "runtime_seconds": f.runtime_seconds,
    }
    for f in progress.families
])
families

### Progress by family

One horizontal bar per family — length is the fraction of expected cells that have landed, colour is the family's job status (green = completed, blue = running, gray = pending, vermillion = failed, amber = partial), and the `done/expected · status` label carries the same status in text (never colour alone). Families keep DAG order, the ensemble node last. This is the waterfall of a run coming in.

In [ ]:
import matplotlib.pyplot as plt

from scale_forecasting.review import plot_progress

if progress.families:
    plot_progress(progress)
    plt.tight_layout()
    plt.show()
else:
    print(f"No family progress for {RUN_ID} (status={progress.status}) — "
          "the run may not have started, or has no config in the registry yet.")

## Review — how did it do?

`review_run(run_id)` reads the leaderboard, the cross-series metric aggregates, and the prediction counts, and returns a `RunReview`: every model best-first in the run's own `decision_metric`, the best model per family and overall, the ensembles, and each ensemble's lift over the best base model. When a run had no backtest (no aggregates) it falls back to the leaderboard, scoring on WAPE.

> Review a run once it's `COMPLETED`. On a run that turned backtest off for speed (the 100k default), the metric panel is empty — the leaderboard and timeline below are still the story; turn backtest on in a demo config to populate the accuracy view.

In [ ]:
from scale_forecasting.review import review_run

review = review_run(RUN_ID, settings=settings)
print(f"{review.run_id} — status={review.status} — decision metric: {review.decision_metric} "
      f"(lower is better) — {review.n_series} series\n")

if review.best_overall is not None:
    b = review.best_overall
    print(f"best overall: {b.model_type} ({b.family}) — {review.decision_metric}={b.score:.4g}")
    print("best per family:")
    for fam, m in review.best_per_family.items():
        print(f"  {fam:<14} {m.model_type:<16} {review.decision_metric}={m.score:.4g}")
else:
    print("no scored base model (this run has no backtest) — see the leaderboard/timeline below.")

In [ ]:
# The full model board: every model (base + ensemble) best-first, with the run's decision-metric
# score, per-cell fit time, and forecast-row count. n_predictions == 0 flags a model that scored
# metadata but produced no forecasts (a fully-failed fit).
board = pd.DataFrame([
    {
        "model_type": m.model_type,
        "family": m.family,
        "is_ensemble": m.is_ensemble,
        "compute_engine": m.compute_engine,
        "n_series": m.n_series,
        review.decision_metric: m.score,
        "median_fit_seconds": m.median_fit_seconds,
        "no_artifact_rate": m.no_artifact_rate,
        "n_predictions": m.n_predictions,
    }
    for m in review.models
])
board

### Leaderboard

Scored models ranked, best (lowest decision-metric error) on top, coloured base vs. ensemble. The score is labelled at each bar end; a legend appears when both base models and ensembles are present. Unscored models (no backtest) are dropped from the chart.

In [ ]:
from scale_forecasting.review import plot_leaderboard

plot_leaderboard(review)
plt.tight_layout()
plt.show()

### Metric distribution across series

The mean hides the tail. This shows each model's spread of the decision metric across every series — a line from the 10th to the 90th cross-series percentile with a dot at the median — read straight off the server-side aggregates, so it holds at 100k+ series without pulling per-series rows. A model can win on the mean and still be wide (unreliable) in the tail; here you see it.

In [ ]:
from scale_forecasting.review import plot_metric_distribution

plot_metric_distribution(review)
plt.tight_layout()
plt.show()

### Ensemble lift

The question every ensemble has to answer: *did combining beat the best single model?* Each ensemble's `lift` is `best_base_score − score` in the decision metric (positive = the ensemble is better, since lower error is better); `lift_pct` is that as a fraction of the base score. Best lift first — a negative lift means the ensemble did worse than just picking the champion.

In [ ]:
if review.ensemble_lift:
    lift = pd.DataFrame([
        {
            "ensemble": e.model_type,
            "score": e.score,
            "best_base_model": e.best_base_model,
            "best_base_score": e.best_base_score,
            "lift": e.lift,
            "lift_pct": e.lift_pct,
        }
        for e in review.ensemble_lift
    ])
    display(lift)
else:
    print("No ensemble lift to report — this run has no scored ensemble (or no scored base model "
          "to compare against).")

## Execution timeline — the family DAG over wall-clock

The waterfall of *when* things ran, reusing the SDK's trace builder. `build_trace_frame` stacks the per-family job spans (`v_run_jobs`) and the per-cell timing brackets (`forecast_metadata`) into one long-form frame; `plot_trace` renders it as a Gantt — each family job on its own lane, the cells that ran under it on worker lanes below. This is where you see a family's job overlap the next, and which family set the run's wall-clock. (Cell rows are capped for a bounded read; the per-job lanes are the whole-run view.)

In [ ]:
from scale_forecasting.registry import bq
from scale_forecasting.sdk import build_trace_frame, plot_trace

job_rows = bq.read_run_jobs(RUN_ID, settings=settings)
cell_rows = bq.read_cell_timing(RUN_ID, settings=settings)
frame = build_trace_frame(job_rows, cell_rows)

if not frame.empty:
    plot_trace(frame, title=f"{RUN_ID} — execution timeline")
    plt.tight_layout()
    plt.show()
else:
    print(f"No timed rows for {RUN_ID} — older runs predate the wall-clock trace columns, or the "
          "run hasn't recorded any job/cell brackets yet.")
frame.head()